# Territorial Digital Divide – Geospatial Raster Analysis
**Cusco, Peru | NASA VNL × Mobile Coverage**

Pipeline measuring digital inequality using nighttime radiance and mobile network density as proxies for urbanization and internet access.

---
## Setup

In [2]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.crs import CRS
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from scipy.ndimage import gaussian_filter
from scipy import stats
import seaborn as sns
import pandas as pd
from pathlib import Path

# Paths
DATA_DIR   = Path('../data')
OUTPUT_DIR = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

VNL_PATH  = DATA_DIR / 'VNL_cusco_2025.tif'
KERN_PATH = DATA_DIR / 'kernel_cobmovil2019_50m.tif'

---
## Steps 0–1 | Load & Inspect Rasters

In [3]:
def inspect_raster(path: Path, label: str) -> dict:
    """Open a raster and print its key metadata."""
    with rasterio.open(path) as src:
        info = {
            'label':      label,
            'crs':        src.crs.to_string(),
            'width':      src.width,
            'height':     src.height,
            'count':      src.count,
            'nodata':     src.nodata,
            'bounds':     src.bounds,
            'res_x':      src.res[0],
            'res_y':      src.res[1],
            'dtype':      src.dtypes[0],
        }
    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")
    print(f"  CRS          : {info['crs']}")
    print(f"  Dimensions   : {info['height']} rows × {info['width']} cols")
    print(f"  Bands        : {info['count']}")
    print(f"  Data type    : {info['dtype']}")
    print(f"  NoData value : {info['nodata']}")
    print(f"  Bounds       : {info['bounds']}")
    print(f"  Pixel res    : {info['res_x']:.6f} × {info['res_y']:.6f} (x, y)")
    return info

vnl_info  = inspect_raster(VNL_PATH,  'VNL_cusco_2025.tif  (NASA Nighttime Radiance)')
kern_info = inspect_raster(KERN_PATH, 'kernel_cobmovil2019_50m.tif  (Mobile Coverage Kernel)')


  VNL_cusco_2025.tif  (NASA Nighttime Radiance)
  CRS          : EPSG:4326
  Dimensions   : 1081 rows × 961 cols
  Bands        : 1
  Data type    : float32
  NoData value : None
  Bounds       : BoundingBox(left=-74.00208248534999, bottom=-15.50208405735, right=-69.99791578664998, top=-10.99791735465)
  Pixel res    : 0.004167 × 0.004167 (x, y)

  kernel_cobmovil2019_50m.tif  (Mobile Coverage Kernel)
  CRS          : EPSG:32719
  Dimensions   : 6116 rows × 7754 cols
  Bands        : 1
  Data type    : float32
  NoData value : -3.4028234663852886e+38
  Bounds       : BoundingBox(left=-43080.11101302641, bottom=8337100.058809407, right=344619.88898697356, top=8642900.058809407)
  Pixel res    : 50.000000 × 50.000000 (x, y)


---
## Step 2 | Reproject & Align Connectivity to VNL Grid

In [4]:
with rasterio.open(VNL_PATH) as vnl_src:
    vnl_meta    = vnl_src.meta.copy()
    vnl_data    = vnl_src.read(1).astype(np.float32)
    vnl_nodata  = vnl_src.nodata
    vnl_crs     = vnl_src.crs
    vnl_transform = vnl_src.transform

with rasterio.open(KERN_PATH) as kern_src:
    kern_crs = kern_src.crs

    # Calculate transform from UTM-19S → WGS84 with VNL dimensions
    dst_transform, dst_width, dst_height = calculate_default_transform(
        kern_crs, vnl_crs,
        kern_src.width, kern_src.height,
        *kern_src.bounds
    )

    # Reproject connectivity to a temporary array in WGS84
    kern_reprojected = np.zeros((vnl_data.shape[0], vnl_data.shape[1]), dtype=np.float32)

    reproject(
        source=rasterio.band(kern_src, 1),
        destination=kern_reprojected,
        src_transform=kern_src.transform,
        src_crs=kern_crs,
        dst_transform=vnl_transform,   # align to VNL grid
        dst_crs=vnl_crs,
        resampling=Resampling.bilinear
    )

print(f"VNL shape           : {vnl_data.shape}")
print(f"Connectivity shape  : {kern_reprojected.shape}")
print(f"Grids aligned       : {vnl_data.shape == kern_reprojected.shape}")

VNL shape           : (1081, 961)
Connectivity shape  : (1081, 961)
Grids aligned       : True
